<img width="8%" alt="Archivist" src="https://upload.wikimedia.org/wikipedia/commons/thumb/e/e1/Biblioth%C3%A8que_Inguimbertine_salle_X.jpg/1920px-Biblioth%C3%A8que_Inguimbertine_salle_X.jpg" style="border-radius: 15%">

# Archivist - Chat
<a href="https://github.com/mbasri/archivist">Give Feedback</a> | <a href="https://github.com/mbasri/archivist">Bug report</a>

**Tags:** #archivist #chat #rag #qdrant #ollama #llamaindex #python

**Author:** [Mohamed BASRI](https://github.com/mbasri)

**Last update:** 2026-04-06 (Created: 2026-04-06)

**Description:** Interactive RAG chatbot — loads the Qdrant vector store, creates a LlamaIndex chat engine with conversation history, and runs an interactive chat loop. Each answer is grounded in your indexed documents, with sources displayed.

**References:**
- [LlamaIndex Chat Engine](https://docs.llamaindex.ai/en/stable/module_guides/deploying/chat_engines/)
- [LlamaIndex Ollama LLM](https://docs.llamaindex.ai/en/stable/examples/llm/ollama/)
- [Ollama](https://ollama.com/)
- [Archivist project](https://github.com/mbasri/archivist)

## Setup

### Install dependencies

In [ ]:
import sys
import subprocess
from pathlib import Path

project_root = Path().resolve()
project_root = project_root.parent if project_root.name == "notebooks" else project_root

result = subprocess.run(
    [sys.executable, "-m", "pip", "install",
     "-r", str(project_root / "requirements.txt")],
    capture_output=True, text=True
)

print(result.stdout[-3000:] if result.stdout else "")
print(result.stderr[-3000:] if result.stderr else "")

if result.returncode != 0:
    print(f"✗ Install failed (exit code {result.returncode})")
else:
    print("✓ Dependencies installed")

## Input

### Import libraries

In [ ]:
import os
from dotenv import load_dotenv

from llama_index.core import VectorStoreIndex
from llama_index.embeddings.ollama import OllamaEmbedding
from llama_index.llms.ollama import Ollama
from llama_index.vector_stores.qdrant import QdrantVectorStore
from qdrant_client import QdrantClient

load_dotenv(project_root / ".env", override=True)
print("✓ .env loaded")

### Setup Variables
- `llm_model`: Ollama model used to generate answers — run `ollama list` to see available models
- `similarity_top_k`: number of document chunks retrieved from Qdrant per query
- `chat_mode`: `context` retrieves fresh chunks at every turn; `condense_plus_context` also rewrites the question using history

In [ ]:
collection_name  = os.getenv("COLLECTION_NAME",  "archivist")
qdrant_host      = os.getenv("QDRANT_HOST",      "localhost")
qdrant_port      = int(os.getenv("QDRANT_PORT",  6333))
ollama_base_url  = os.getenv("OLLAMA_BASE_URL",  "http://localhost:11434")
embedding_model  = os.getenv("EMBEDDING_MODEL",  "nomic-embed-text")
llm_model        = os.getenv("LLM_MODEL",        "llama3.2:1b")
similarity_top_k = int(os.getenv("SIMILARITY_TOP_K", 5))

embed_model = OllamaEmbedding(model_name=embedding_model, base_url=ollama_base_url)
llm         = Ollama(model=llm_model, base_url=ollama_base_url, request_timeout=300.0)

print(f"Qdrant    : {qdrant_host}:{qdrant_port} / {collection_name}")
print(f"Embedding : {embedding_model}")
print(f"LLM       : {llm_model}")
print(f"Top-K     : {similarity_top_k}")

## Model

### Connect to Qdrant and load index

In [ ]:
qdrant_client = QdrantClient(host=qdrant_host, port=qdrant_port)
vector_store  = QdrantVectorStore(client=qdrant_client, collection_name=collection_name)
index         = VectorStoreIndex.from_vector_store(vector_store, embed_model=embed_model)

info = qdrant_client.get_collection(collection_name)
print(f"Connected — {info.points_count} vectors in '{collection_name}'")

### Create chat engine
LlamaIndex maintains the conversation history and injects relevant document chunks as context at each turn.
Re-run this cell to reset the conversation.

In [ ]:
chat_engine = index.as_chat_engine(
    llm=llm,
    similarity_top_k=similarity_top_k,
    chat_mode="context",
    verbose=False,
)
print("✓ Chat engine ready")

## Output

### Chat
Edit `question` and re-run this cell for each message. The chat engine keeps conversation history between runs.
Re-run the **Create chat engine** cell above to reset the conversation.

In [ ]:
question = "Bonjour"

response = chat_engine.chat(question)
print(f"You: {question}")
print(f"\nArchivist: {response.response}\n")

if response.source_nodes:
    seen = set()
    sources = []
    for node in response.source_nodes:
        name = node.metadata.get("file_name", "unknown")
        if name not in seen:
            sources.append(f"  - {name}  (score: {node.score:.3f})")
            seen.add(name)
    print("Sources:")
    print("\n".join(sources))